*This is the notebook for the training phase of this example*

## Recurrent Neural Network Demonstration
**Goal:** Predict the function $f(x + \frac{\pi}{50})$ from $f(x)$

**Strategy:**
1. Architecture: 1 explicit state + 3 hidden state
2. The explicit state will be at slot 0, hidden state will be at slot [1, 2, 3]
2. We train in batch of 10, train the network to go forward 100 steps.
4. However, to prevent Gradient Explosion/Vanishing, we will train 10 steps at a time.
3. We use Adam optimizer to optimize.
4. We save our training result to a `.cache` file.

This library works similarly to Pytorch, and I'm proud of that.

### 1. Import and Config

In [1]:
import math
import lktorch as lk
import os

CURRENT_DIR = os.getcwd()
WEIGHT_PATH = os.path.join(CURRENT_DIR, "training_data", "model_weight.cache")

### 2. Generate and preprocess

In [2]:
# 10 data points (10 different starting x), each with 100 forward steps
public_test = []
for i in range(0, 100):
    current_test = []
    for j in range(0, 10):
        current_test.append(math.sin(math.pi * (i + j) / 50))
    public_test.append(current_test)

print("Done fetching data!")

Done fetching data!


### 3. Helper Functions

In [3]:

def join_info(public_state, hidden_state):
    sz = len(public_state.dimension())
    perm = [sz-1]
    for i in range(sz-1): 
        perm.append(i)
    public_state = public_state.PermuteDimension(perm)
    hidden_state = hidden_state.PermuteDimension(perm)
    state = lk.Merge(public_state, hidden_state)

    invperm = []
    for i in range(sz - 1): 
        invperm.append(i + 1)
    invperm.append(0)
    return state.PermuteDimension(invperm)

def extract_public(tensor):
    return tensor.Slice([0], [0])

def extract_hidden(tensor):
    return tensor.Slice([1], [3])

def calculate_loss(model):
    input_data = lk.Tensor(public_test[0])
    dim = input_data.dimension()
    input_data = input_data.Reshape(dim + [1])
    input_data = join_info(input_data, lk.Zeros(dim + [3]))

    cur_tensor = input_data
    cum_loss = lk.Zeros([])
    for it in range(1, len(public_test)):
        cur_tensor = model(cur_tensor)
        public_state = extract_public(cur_tensor)
        hidden_state = extract_hidden(cur_tensor)
        
        test_state = lk.Tensor(public_test[it]).Reshape(dim + [1])
        cum_loss = cum_loss + lk.MeanAll(lk.MSELoss(public_state, test_state))
        cur_tensor = join_info(test_state, hidden_state)
    cum_loss = cum_loss + lk.SumAll(cur_tensor) * 0
    return cum_loss

### 4. Training

In [4]:
model = lk.Sequential([
    lk.LinearLayer(4, 8),
    lk.Tanh_Layer(),
    lk.LinearLayer(8, 8),
    lk.Tanh_Layer(),
    lk.LinearLayer(8, 4),
    lk.Tanh_Layer(),
    lk.ScalarMultiply_Layer(float(1.2))
])

optimizer = lk.Adam(0.001, 0.9, 0.999)
optimizer.add_parameter(model.get_parameters())


for epoch in range(1, 501):
    for segment in range(0, 10):
        l = 1
        if segment > 0:
            l = segment * 10
        r = segment * 10 + 9

        optimizer.zero_gradient()

        input_data = lk.Tensor(public_test[0])
        dim = input_data.dimension()
        input_data = input_data.Reshape(dim + [1])
        input_data = join_info(input_data, lk.Zeros(dim + [3]))

        cur_tensor = input_data
        cum_loss = lk.Zeros([])

        for it in range(1, l):
            cur_tensor = model(cur_tensor)
            public_state = extract_public(cur_tensor)
            hidden_state = extract_hidden(cur_tensor)
            
            test_state = lk.Tensor(public_test[it]).Reshape(dim + [1])
            cur_tensor = join_info(test_state, hidden_state)
        for it in range(l, r+1):
            cur_tensor = model(cur_tensor)
            public_state = extract_public(cur_tensor)
            hidden_state = extract_hidden(cur_tensor)
            
            test_state = lk.Tensor(public_test[it]).Reshape(dim + [1])
            cum_loss = cum_loss + lk.MeanAll(lk.MSELoss(public_state, test_state))
            cur_tensor = join_info(test_state, hidden_state)
        cum_loss = cum_loss + lk.SumAll(cur_tensor) * 0
        cum_loss.backward()
        optimizer.step()

    if (epoch % 10 == 9):
        print("Epoch: ", epoch)
        print("--- Loss: ", calculate_loss(model))

Epoch:  9
--- Loss:  9.02047
Epoch:  19
--- Loss:  1.91438
Epoch:  29
--- Loss:  0.942981
Epoch:  39
--- Loss:  0.766035
Epoch:  49
--- Loss:  0.666111
Epoch:  59
--- Loss:  0.601218
Epoch:  69
--- Loss:  0.550818
Epoch:  79
--- Loss:  0.511488
Epoch:  89
--- Loss:  0.474063
Epoch:  99
--- Loss:  0.443609
Epoch:  109
--- Loss:  0.418107
Epoch:  119
--- Loss:  0.39601
Epoch:  129
--- Loss:  0.376273
Epoch:  139
--- Loss:  0.358234
Epoch:  149
--- Loss:  0.341654
Epoch:  159
--- Loss:  0.32641
Epoch:  169
--- Loss:  0.312232
Epoch:  179
--- Loss:  0.298932
Epoch:  189
--- Loss:  0.286367
Epoch:  199
--- Loss:  0.274435
Epoch:  209
--- Loss:  0.263073
Epoch:  219
--- Loss:  0.252231
Epoch:  229
--- Loss:  0.241876
Epoch:  239
--- Loss:  0.231985
Epoch:  249
--- Loss:  0.222548
Epoch:  259
--- Loss:  0.21356
Epoch:  269
--- Loss:  0.205019
Epoch:  279
--- Loss:  0.196924
Epoch:  289
--- Loss:  0.18927
Epoch:  299
--- Loss:  0.18205
Epoch:  309
--- Loss:  0.175248
Epoch:  319
--- Loss:  0.1

### 5. Save to file

In [5]:
lk.WriteFile(model.get_state_dict(), WEIGHT_PATH)
print("File saved successfully!")

File saved successfully!
